<a href="https://colab.research.google.com/github/tomo-mar/project_research_A/blob/main/highlight_estimated_VAD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1, 環境設定

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from google.colab import drive

drive.mount('/content/drive')

# 音声分離の有無
USE_VOCAL_ISOLATED = True

BASE_DIR = "/content/drive/MyDrive/VAD_Experiment"
INPUT_VAD_DIR = f"{BASE_DIR}/outputs_isolated" if USE_VOCAL_ISOLATED else f"{BASE_DIR}/outputs_mixed"
INPUT_LABEL_DIR = f"{BASE_DIR}/labels" # 正解ラベル
MODEL_OUT_DIR = f"{BASE_DIR}/models"

os.makedirs(MODEL_OUT_DIR, exist_ok=True)
print(f"モード: {'音声分離版' if USE_VOCAL_ISOLATED else '通常版'}")

2, データ前処理

In [ ]:
def load_and_preprocess_data(video_ids):
    X_list = []
    y_list = []

    for vid in video_ids:
        vad_path = f"{INPUT_VAD_DIR}/{vid}_VAD.csv"
        if not os.path.exists(vad_path):
            continue

        df_vad = pd.read_csv(vad_path, header=None, index_col=0).T
        df_vad.columns = ["Time", "V", "A", "D"]
        df_vad = df_vad.apply(pd.to_numeric)

        label_path = f"{INPUT_LABEL_DIR}/{vid}_Label.csv"
        if not os.path.exists(label_path):
            continue

        df_label = pd.read_csv(label_path)
        df_merged = pd.merge(df_vad, df_label, on="Time", how="inner")

        # 特徴量追加
        df_merged["V_abs"] = abs(df_merged["V"] - 5.0)
        X_list.append(df_merged[["V", "A", "D", "V_abs"]].values)
        y_list.append(df_merged["Highlight_Prob"].values)

    if not X_list:
        return np.array([]), np.array([])

    X = np.vstack(X_list)
    y = np.concatenate(y_list)
    return X, y

print("前処理関数の定義完了")

3, 学習データ構築

In [ ]:
# ファイル名の指定
train_videos = ["video01", "video02", "video03", "video04", "video05"]
test_videos = ["video06", "video07", "video08", "video09", "video10"]

X_train, y_train = load_and_preprocess_data(train_videos)
X_test, y_test = load_and_preprocess_data(test_videos)

if len(X_train) == 0:
    print("データが読み込めませんでした")
else:
    print(f"学習用データ: {X_train.shape[0]} チャンク")
    print(f"テスト用データ: {X_test.shape[0]} チャンク")

4, モデル学習・パラメータ解析

In [ ]:
import matplotlib.pyplot as plt
from sklearn.inspection import PartialDependenceDisplay

model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)

# 1. 特徴量の重み付け
importances = model.feature_importances_
print("\n📊 【特徴量の重み】")
print(f"  * Valence (生データ) : {importances[0]:.3f}")
print(f"  * Arousal            : {importances[1]:.3f}")
print(f"  * Dominance          : {importances[2]:.3f}")
print(f"  * Valence (|V-5|)    : {importances[3]:.3f}")

print("\n各パラメータとハイライト確率の関係")

fig, ax = plt.subplots(figsize=(12, 4))
feature_names = ["Valence(Raw)", "Arousal", "Dominance", "Valence(|V-5|)"]

display = PartialDependenceDisplay.from_estimator(
    model,
    X_train,
    features=[0, 1, 2, 3],
    feature_names=feature_names,
    ax=ax
)

plt.tight_layout()
plt.show()

5, テストデータ推論

In [ ]:
import matplotlib.pyplot as plt
import os
import pandas as pd
from sklearn.metrics import mean_squared_error

OUTPUT_PRED_DIR = f"{BASE_DIR}/predictions_{'isolated' if USE_VOCAL_ISOLATED else 'mixed'}"
os.makedirs(OUTPUT_PRED_DIR, exist_ok=True)

print("テストデータ実行\n")

total_mse = 0
valid_videos = 0

for vid in test_videos:
    vad_path = f"{INPUT_VAD_DIR}/{vid}_VAD.csv"
    label_path = f"{INPUT_LABEL_DIR}/{vid}_Label.csv"

    if not os.path.exists(vad_path) or not os.path.exists(label_path):
        print(f"{vid} のデータが揃っていないためスキップ")
        continue

    df_vad = pd.read_csv(vad_path, header=None, index_col=0).T
    df_vad.columns = ["Time", "V", "A", "D"]
    df_vad = df_vad.apply(pd.to_numeric)

    df_label = pd.read_csv(label_path)
    df_merged = pd.merge(df_vad, df_label, on="Time", how="inner")

    if len(df_merged) == 0:
        continue

    # 特徴量追加
    df_merged["V_abs"] = abs(df_merged["V"] - 5.0)

    X_vid = df_merged[["V", "A", "D", "V_abs"]].values
    y_vid_true = df_merged["Highlight_Prob"].values
    time_vid = df_merged["Time"].values

    y_vid_pred = model.predict(X_vid)

    mse = mean_squared_error(y_vid_true, y_vid_pred)
    total_mse += mse
    valid_videos += 1
    print(f"{vid} - テスト誤差 (MSE): {mse:.4f}")

    df_out = pd.DataFrame({
        "Time": time_vid,
        "True_Prob": y_vid_true,
        "Pred_Prob": y_vid_pred
    })
    csv_out_path = f"{OUTPUT_PRED_DIR}/{vid}_Prediction.csv"
    df_out.to_csv(csv_out_path, index=False)

    # 比較グラフの描画
    plt.figure(figsize=(10, 4))

    # 人間の正解ラベル (青)
    plt.plot(time_vid, y_vid_true, label="True Highlight Prob (Human)", color="blue", marker="o")
    # AIの予測ラベル (赤)
    plt.plot(time_vid, y_vid_pred, label="Predicted Prob (AI)", color="red", linestyle="--", marker="x")

    plt.title(f"Highlight Probability Tracking: {vid} ({'Isolated' if USE_VOCAL_ISOLATED else 'Mixed'})")
    plt.xlabel("Time [s]")
    plt.ylabel("Highlight Probability")
    plt.ylim(0, 1.05)
    plt.legend()
    plt.grid(True, linestyle=":", alpha=0.7)
    plt.tight_layout()
    plt.show()

if valid_videos > 0:
    print(f"\n全テスト動画の平均 MSE: {total_mse / valid_videos:.4f}")
    print(f"予測結果CSVを {OUTPUT_PRED_DIR} に保存")